# Data quality and cleaning

This synthetic subscription dataset contains deliberately injected defects. We repair deterministic fields and quarantine ambiguous financial/date records. Cancellation is the only churn definition.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / 'scripts').exists():
    root = root.parent
assert (root / 'scripts').exists(), 'Open this notebook inside the project'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
from scripts.common import ROOT, AS_OF, read_csv
from IPython.display import display, Image


In [2]:
raw = pd.read_csv(ROOT / 'data/raw/customers.csv')
raw.shape, raw.customer_id.nunique(), raw.head()

((12120, 26),
 12000,
   customer_id subscription_start_date signup_date plan_type contract_type  \
 0     C000001              2025-03-09  2025-02-23   Premium       Monthly   
 1     C000002              2025-07-03  2025-06-21     Basic       Monthly   
 2     C000003              2025-05-13  2025-05-09  Standard       Monthly   
 3     C000004              2025-11-08  2025-11-05     Basic       Monthly   
 4     C000005              2025-02-08  2025-02-07     Basic       Monthly   
 
    monthly_fee  payment_method  auto_renew acquisition_channel     city  ...  \
 0        109.0   Bank Transfer           1             Organic  Seattle  ...   
 1         29.0  Digital Wallet           1              Social   Boston  ...   
 2         59.0   Bank Transfer           1             Organic  Detroit  ...   
 3         29.0  Digital Wallet           1             Organic   Denver  ...   
 4         29.0   Bank Transfer           1             Organic  Chicago  ...   
 
   login_frequency m

## Audit the cleaning policy

Exact duplicates can be removed safely. Conflicting identities or invalid fees/dates require quarantine. We do not invent replacement monthly fees or cancellation dates.

In [3]:
from scripts.data_cleaning import clean_customers
clean, quarantine, repairs, audit = clean_customers(raw)
display(pd.Series({k:v for k,v in audit.items() if not isinstance(v,dict)}))
display(quarantine.quarantine_reason.value_counts())

raw_customer_rows                                                                    12120
exact_duplicate_customer_rows_removed                                                  120
quarantined_customers                                                                  120
clean_customers                                                                      11880
repair_events                                                                          210
policy                                   Exact duplicates removed. Conflicting IDs and ...
dtype: object

quarantine_reason
invalid_churn_date;conflicting_end_and_churn_dates                                              30
invalid_discount                                                                                30
invalid_monthly_fee                                                                             30
invalid_subscription_start;invalid_last_activity                                                21
invalid_subscription_start;invalid_subscription_end;invalid_churn_date;invalid_last_activity     9
Name: count, dtype: int64

In [4]:
display(repairs.groupby(['field','rule']).size().rename('repair_events').reset_index())
assert len(raw) == audit['exact_duplicate_customer_rows_removed'] + len(quarantine) + len(clean)
assert clean.customer_id.is_unique

,field,rule,repair_events
0,churn_flag,validated_end_date_authoritative,30
1,customer_status,validated_end_date_authoritative,30
2,payment_method,missing_category_to_unknown,30
3,plan_type,standardize_plan,30
4,region,city_reference_mapping,30
5,tenure_months,recompute_tenure_from_dates,60


## Review derived customer attributes

Tenure ends at cancellation for churned customers. Observed CLV is earned historical revenue. Risk scores apply to active customers only.

In [5]:
customers = read_csv('customers_clean')
display(customers[['customer_id','tenure_months','engagement_score','observed_clv','risk_band','customer_segment']].head(10))
assert customers.loc[customers.churn_flag.eq(1),'risk_score'].isna().all()

,customer_id,tenure_months,engagement_score,observed_clv,risk_band,customer_segment
0,C000001,9.0349,46.7070,984.52,Not Applicable,Churned High Value
1,C000002,7.6879,48.3386,201.53,Not Applicable,Churned Low Value
2,C000003,1.9713,40.2007,106.20,Not Applicable,Churned Low Value
3,C000004,9.7248,43.0874,269.07,Low Risk,Core Customers
4,C000005,4.6982,33.5128,138.69,Not Applicable,Churned Low Value
5,C000006,4.8296,45.0777,135.08,Not Applicable,Churned Low Value
6,C000007,1.7413,38.6077,45.46,Low Risk,New Customers
7,C000008,15.5400,49.2341,360.72,Medium Risk,Core Customers
8,C000009,9.7248,68.5048,283.23,Low Risk,Highly Engaged
9,C000010,2.6283,19.1471,155.43,Not Applicable,Churned Low Value


## Interpretation

The quarantine population is excluded consistently from customer and monthly facts. See outputs/data_quality_report.json and data/quarantine/repair_log.csv for row accounting and field-level provenance.